1. Carga de datos:

In [1]:
import importlib.util
import sys
import os

# Definir ruta física
path_to_file = os.path.abspath('../src/retail-sales-analysis_P3.py')

# Cargar módulo manualmente
spec = importlib.util.spec_from_file_location("retail_module", path_to_file)
retail_module = importlib.util.module_from_spec(spec)
sys.modules["retail_module"] = retail_module
spec.loader.exec_module(retail_module)

# extraer función
cargar_datos = retail_module.cargar_datos

print("¡Conexión establecida con el archivo .py!")

# Definir ruta del dataset y cargar Dataframe
ruta_archivo = '../data/retail_sales_dataset.csv'
df = cargar_datos(ruta_archivo) # Definir Dataframe

¡Conexión establecida con el archivo .py!


2. Exploración inicial:

In [2]:
# Inspeccionar dimensiones y tipos de datos
print(f'Dataset cargado exitosamente!')
print(f'Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas')
print(f'Memoria aproximada: {df.memory_usage(deep=True).sum() / 1024:.0f} KB')
print(f'\nColumnas: {list(df.columns)}')

Dataset cargado exitosamente!
Dimensiones: 1000 filas x 9 columnas
Memoria aproximada: 260 KB

Columnas: ['Transaction ID', 'Date', 'Customer ID', 'Gender', 'Age', 'Product Category', 'Quantity', 'Price per Unit', 'Total Amount']


4. Chequear extremos del dataset

In [3]:
# Visualizar primeros y últimos lugares en el dataset
print("\nPrimeros 5 registros:")
print(df.head(5))
print("\nÚltimos 5 registros:")
print(df.tail(5))


Primeros 5 registros:
   Transaction ID        Date Customer ID  Gender  Age Product Category  \
0               1  2023-11-24     CUST001    Male   34           Beauty   
1               2  2023-02-27     CUST002  Female   26         Clothing   
2               3  2023-01-13     CUST003    Male   50      Electronics   
3               4  2023-05-21     CUST004    Male   37         Clothing   
4               5  2023-05-06     CUST005    Male   30           Beauty   

   Quantity  Price per Unit  Total Amount  
0         3              50           150  
1         2             500          1000  
2         1              30            30  
3         1             500           500  
4         2              50           100  

Últimos 5 registros:
     Transaction ID        Date Customer ID  Gender  Age Product Category  \
995             996  2023-05-16     CUST996    Male   62         Clothing   
996             997  2023-11-17     CUST997    Male   52           Beauty   
997      

In [4]:
print("=== 5 últimas filas ===")
print()
display(df.info(5))
print("No se encontraron valores nulos.")
print("================================")

=== 5 últimas filas ===

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction ID    1000 non-null   int64
 1   Date              1000 non-null   str  
 2   Customer ID       1000 non-null   str  
 3   Gender            1000 non-null   str  
 4   Age               1000 non-null   int64
 5   Product Category  1000 non-null   str  
 6   Quantity          1000 non-null   int64
 7   Price per Unit    1000 non-null   int64
 8   Total Amount      1000 non-null   int64
dtypes: int64(5), str(4)
memory usage: 70.4 KB


None

No se encontraron valores nulos.


In [5]:
print("=== Estadísticas descriptivas ===")
print()
display(df.describe().round(2))
print("=================================")

=== Estadísticas descriptivas ===



,Transaction ID,Age,Quantity,Price per Unit,Total Amount
count,1000.00,1000.00,1000.00,1000.00,1000.0
mean,500.50,41.39,2.51,179.89,456.0
std,288.82,13.68,1.13,189.68,560.0
min,1.00,18.00,1.00,25.00,25.0
25%,250.75,29.00,1.00,30.00,60.0
50%,500.50,42.00,3.00,50.00,135.0
75%,750.25,53.00,4.00,300.00,900.0
max,1000.00,64.00,4.00,500.00,2000.0


### Desarrollo de instrucciones

1. Transformación de Datos
    * Crea nuevas columnas: Basándonos en los datos existentes, crea nuevas columnas que sean útiles para el análisis. Por ejemplo, calcula el ingreso total por venta y normaliza las ventas.
    * Clasifica los datos: Crea una columna que clasifique las ventas en categorías significativas (e.g., ‘Alta’, ‘Media’, ‘Baja’).


* Nuevas Columnas: 'Ingreso Total' y 'Ventas Normalizadas'

In [7]:
# Se define columna de Ingreso Total
df['Ingreso Total'] = df['Price per Unit'] * df['Quantity']

# Normalizar las ventas (Ingreso Total)
# Valores al rango [0, 1]
min_ingreso = df['Ingreso Total'].min()
max_ingreso = df['Ingreso Total'].max()

df['Ventas Normalizadas'] = ((df['Ingreso Total'] - min_ingreso) / (max_ingreso - min_ingreso)).round(2)

# Verificar nuevas columnas
print("Nuevas columnas creadas en el DataFrame 'df':")
display(df[['Price per Unit', 'Quantity', 'Ingreso Total', 'Ventas Normalizadas']].head())

Nuevas columnas creadas en el DataFrame 'df':


,Price per Unit,Quantity,Ingreso Total,Ventas Normalizadas
0,50,3,150,0.06
1,500,2,1000,0.49
2,30,1,30,0.00
3,500,1,500,0.24
4,50,2,100,0.04


* Calsificar Ventas

** Ver montos de ventas y su frecuencia


In [8]:
print("=== Montos de Ventas y su frecuencia ===")
print()
Contar_ventas = df['Ingreso Total'].value_counts().sort_index(ascending=False)
print(Contar_ventas)
print("Se definen las categorías de ventas:")
print("* Alta > 1000")
print("* 1000 >= Media > 100")
print("* 100 >= Baja")
print("========================================")


=== Montos de Ventas y su frecuencia ===

Ingreso Total
2000     49
1500     50
1200     54
1000     49
900      62
600      35
500      51
300      46
200      62
150      42
120      43
100     108
90       44
75       43
60       45
50      115
30       51
25       51
Name: count, dtype: int64
Se definen las categorías de ventas:
* Alta > 1000
* 1000 >= Media > 100
* 100 >= Baja


** Ejecutar función para categorizar las ventas

In [9]:
clasificar_ventas = retail_module.clasificar_ventas

# Ejecutar la clasificación
df = clasificar_ventas(df)

# Ver el resultado
print(df[['Ingreso Total', 'Categoria Venta']].head())

   Ingreso Total Categoria Venta
0            150           Media
1           1000           Media
2             30            Baja
3            500           Media
4            100            Baja



2. Agrupación y Agregación
    * Agrupación por múltiples columnas: Realiza agrupaciones por categorías como producto y tienda, producto y mes, etc.
    * Aplicar funciones de agregación: Utiliza funciones como sum, mean, count, min, max, std, y var para obtener estadísticas descriptivas de cada grupo.


* Agrupación por Categoría de Producto y Género

In [11]:
# Agrupación por Producto y Género
agrupado_genero = df.groupby(['Product Category', 'Gender'])['Total Amount'].agg(['sum', 'mean', 'count', 'min', 'max', 'std', 'var'])

print("Estadísticas por Producto y Género:")
display(agrupado_genero.round(2))

Estadísticas por Producto y Género:


sum    mean  count  min   max     std        var
Product Category Gender                                                    
Beauty           Female  74830  450.78    166   25  2000  538.74  290235.44
                 Male    68685  487.13    141   25  2000  592.90  351530.08
Clothing         Female  81275  467.10    174   25  2000  577.02  332948.03
                 Male    74305  419.80    177   25  2000  524.12  274697.83
Electronics      Female  76735  451.38    170   25  2000  548.64  301010.95
                 Male    80170  466.10    172   25  2000  587.13  344721.29

* Agrupación por Categoría de Producto y Mes

In [15]:
import pandas as pd

# Convertir a formato fecha y extraer mes
df['Date'] = pd.to_datetime(df['Date'])
df['Mes'] = df['Date'].dt.month

# Agrupación por Producto y Mes
agrupado_mes = df.groupby(['Product Category', 'Mes'])['Total Amount'].agg(['sum', 'mean', 'count', 'min', 'max', 'std', 'var'])

print("Estadísticas por Producto y Mes:")
display(agrupado_mes.round(2))

Estadísticas por Producto y Mes:


sum    mean  count  min   max     std        var
Product Category Mes                                                    
Beauty           1    13930  535.77     26   25  2000  658.82  434041.38
                 2    14035  539.81     26   25  2000  697.75  486860.96
                 3    10545  502.14     21   25  1500  482.91  233206.43
                 4    11905  410.52     29   25  2000  519.65  270036.33
                 5    12450  444.64     28   30  2000  571.56  326675.79
                 6    10995  439.80     25   25  2000  512.25  262396.83
                 7    16090  595.93     27   30  2000  626.06  391955.84
                 8     9790  407.92     24   25  1200  451.73  204060.69
                 9     6320  316.00     20   25  2000  505.96  255998.95
                 10   15355  495.32     31   25  2000  552.58  305339.89
                 11    9700  388.00     25   25  2000  583.13  340039.58
                 12   12400  496.00     25   25  2000  583.34  340281.25
Clothing         1    13125  504.81     26   25  2000  540.87  292542.96
                 2    14560  441.21     33   25  2000  551.72  304396.92
                 3    15065  396.45     38   25  2000  497.56  247568.79
                 4    13940  387.22     36   25  2000  576.18  331987.78
                 5    17455  471.76     37   25  2000  567.65  322222.52
                 6    10170  363.21     28   25  1500  437.64  191528.17
                 7     8250  434.21     19   25  2000  602.11  362539.62
                 8    12455  389.22     32   25  2000  564.53  318692.11
                 9     9975  498.75     20   50  2000  630.42  397431.25
                 10   13315  443.83     30   25  2000  617.89  381790.83
                 11   15200  584.62     26   25  2000  565.79  320121.85
                 12   12070  464.23     26   30  2000  540.91  292585.38
Electronics      1     9925  381.73     26   25  2000  568.82  323559.88
                 2    15465  594.81     26   30  2000  623.95  389312.96
                 3     3380  241.43     14   30  1000  277.68   77105.49
                 4     8025  382.14     21   25  1500  512.22  262366.43
                 5    23245  581.12     40   25  2000  653.75  427387.80
                 6    15550  647.92     24   30  2000  675.03  455660.69
                 7    11125  427.88     26   30  2000  507.79  257852.35
                 8    14715  387.24     38   25  2000  509.45  259537.43
                 9     7325  293.00     25   30  2000  436.81  190804.17
                 10   17910  511.71     35   25  2000  662.39  438757.27
                 11   10020  371.11     27   25  1500  505.73  255758.33
                 12   20220  505.50     40   25  2000  560.32  313953.59

* Agrupación por Categoría de producto y rango Etario

In [ ]:
# Crear rangos de edad (bins)
df['Rango_Etario'] = pd.cut(df['Age'], bins=[0, 25, 45, 100], labels=['Joven', 'Adulto', 'Senior'])

# Agrupación por Producto y Rango Etario
agrupado_edad = df.groupby(['Product Category', 'Rango_Etario'])['Total Amount'].agg(['sum', 'mean', 'count', 'min', 'max', 'std', 'var'])

print("Estadísticas por Producto y Rango Etario:")
display(agrupado_edad.round(2))

Estadísticas por Producto y Rango Etario:


sum    mean  count  min   max     std  \
Product Category Rango_Etario                                            
Beauty           Joven         31280  521.33     60   25  2000  619.75   
                 Adulto        59645  492.93    121   25  2000  593.32   
                 Senior        52590  417.38    126   25  2000  503.83   
Clothing         Joven         26510  519.80     51   25  2000  615.27   
                 Adulto        69525  463.50    150   25  2000  562.60   
                 Senior        59545  396.97    150   25  2000  514.05   
Electronics      Joven         26760  461.38     58   25  2000  573.94   
                 Adulto        61180  449.85    136   25  2000  581.51   
                 Senior        68965  465.98    148   25  2000  555.68   

                                     var  
Product Category Rango_Etario             
Beauty           Joven         384086.33  
                 Adulto        352023.61  
                 Senior        253849.49  
Clothing         Joven         378562.96  
                 Adulto        316516.36  
                 Senior        264243.59  
Electronics      Joven         329412.10  
                 Adulto        338158.50  
                 Senior        308775.05


3. Análisis Personalizado con apply
    * Función personalizada: Aplica funciones personalizadas para realizar análisis específicos que no se pueden lograr con las funciones de agregación estándar.
    * Ejemplo de uso avanzado: Calcula la desviación de cada venta respecto a la media de su grupo.
    


4. Documentación
    *  Comentarios claros: Documenta claramente cada paso del análisis, explicando qué se hizo y por qué se hizo.
    * Código legible: Asegúrate de que el código sea legible y esté bien comentado.

5. Chequear si existen datos faltantes

7. Chequear "Falsos Nulos" (NaN, None, "null")

6. Chequear outliers

In [ ]:
# Extraer los montos totales de la columna 8
montos_ventas = []
for fila in datos:
    # fila[8] es el "Total Amount" (Monto Total de la Venta)
    valor_venta = float(fila[8]) 
    montos_ventas.append(valor_venta)

# Se crea un arreglo para trabajar con Numpy
ventas_array = np.array(montos_ventas)

# Encontrar los puntos de corte (Cuartiles)
q1 = np.percentile(ventas_array, 25) # El 25% de las ventas más bajas
q3 = np.percentile(ventas_array, 75) # El 75% de las ventas (donde empiezan las ventas altas)

# Calcular el "Ancho de la Venta Normal" (IQR)
rango_ventas = q3 - q1

# Establecer los límites de lo que aceptamos como "Venta Común"
limite_superior = q3 + (1.5 * rango_ventas)
limite_inferior = q1 - (1.5 * rango_ventas)

# Buscar las ventas que se salen de los límites
ventas_fuera_de_rango = []
for i in ventas_array:
    # Si la venta es mayor al límite superior o menor al inferior
    if i > limite_superior or i < limite_inferior:
        ventas_fuera_de_rango.append(i)

# Mostrar resultados
print("--- ANÁLISIS DE VENTAS TOTALES (OUTLIERS) ---")
print(f"Límite máximo para una venta normal: ${limite_superior:.2f}")
print(f"Límite mínimo para una venta normal: ${limite_inferior:.2f}")
print(f"Cantidad de ventas detectadas como atípicas: {len(ventas_fuera_de_rango)}")

if len(ventas_fuera_de_rango) > 0:
    print(f"Algunas de estas ventas fueron: {ventas_fuera_de_rango[:5]}")

### Nota: Interpretación del Límite Inferior Negativo

Al realizar el análisis de valores atípicos (outliers) mediante el método del Rango Intercuartiles (IQR), se observa que el **Límite Inferior** arroja un valor negativo. A continuación, se explica por qué esto es correcto y qué significa para nuestro análisis:

* **¿Por qué es negativo?**: El cálculo es puramente matemático ($Q1 - 1.5 \times IQR$). Si la dispersión de los precios es muy grande comparada con los valores bajos, la fórmula resta más de lo que hay, cruzando la barrera del cero.
* **Sentido de Realidad**: En el contexto de este dataset de Retail, los montos totales de venta no pueden ser menores a cero.
* **Conclusión**: Un límite inferior negativo no es un error. Simplemente nos confirma que **no existen ventas atípicas por ser "demasiado bajas"**. Todas las ventas pequeñas del dataset entran dentro del rango considerado como "normal".

7 Limpieza de datos

In [ ]:
# Aunque no haya nulos hoy, esta función es para maximizar la seguridad
datos = limpiar_datos(datos) 
print("Pipeline: Datos verificados y listos para el análisis.")

7. Exporar si existen duplicados

In [ ]:
print("--- BUSCANDO REGISTROS REPETIDOS ---")

filas_vistas = []
duplicados_encontrados = []

for fila in datos:
    # Convertir la fila a una lista
    fila_lista = list(fila)
    
    if fila_lista in filas_vistas:
        duplicados_encontrados.append(fila_lista)
    else:
        filas_vistas.append(fila_lista)

print(f"Cantidad de filas duplicadas: {len(duplicados_encontrados)}")

if len(duplicados_encontrados) > 0:
    print("Se encontraron datos repetidos. Es necesario eliminarlos para no alterar las estadísticas.")

7. Exploración de datos

In [ ]:
categorias_analisis = ['Beauty', 'Clothing', 'Electronics']

# Listas para guardar los resultados
nombres_cats = []
resultados_totales = []

print("--- ANÁLISIS DE VENTAS DIARIAS POR CATEGORÍA ---")

for cat in categorias_analisis:
    # Filtrar los datos por categoría
    datos_cat = filtrar_por_categoria(datos, cat)
    
    if len(datos_cat) > 0:
        # Crear un diccionario para agrupar ventas por fecha
        ventas_por_dia = {}
        
        for fila in datos_cat:
            fecha = fila[1] # Columna 1 es la fecha
            monto = float(fila[8]) # Columna 8 es el monto
            
            # Si el día ya existe, suma monto al total del día
            if fecha in ventas_por_dia:
                ventas_por_dia[fecha] = ventas_por_dia[fecha] + monto
            # Si no existe, se agrega
            else:
                ventas_por_dia[fecha] = monto
        
        # Declarar lista con los totales de cada día
        totales_diarios = list(ventas_por_dia.values())
        
        # Calcular promedio diario
        promedio_diario = np.mean(totales_diarios)
        total_categoria = np.sum(totales_diarios)
        
        # Guardar nombre y total para comparar al final
        nombres_cats.append(cat)
        resultados_totales.append(total_categoria)
        
        print(f"\nCategoría: {cat}")
        print(f"  - Total acumulado: ${total_categoria:,.2f}")
        print(f"  - Promedio de ventas diarias: ${promedio_diario:,.2f}")
        print(f"  - Cantidad de días con ventas: {len(totales_diarios)}")

# --- IDENTIFICACIÓN DE MAYORES Y MENORES ---
print("\n--- IDENTIFICACIÓN FINAL ---")

# valores extremos basados en el total acumulado
mayor_v = max(resultados_totales)
menor_v = min(resultados_totales)

# Categorías correspondientes almax y min de ventas totales
cat_mayor = nombres_cats[resultados_totales.index(mayor_v)]
cat_menor = nombres_cats[resultados_totales.index(menor_v)]

# Resultado final
print(f"Tras el análisis, la categoría con mayores ventas es '{cat_mayor}' y la categoría con menores ventas es '{cat_menor}'.")

8. Manipulación de Datos

In [ ]:
# Mostrar cuántos registros quedan post limpieza
print(f"Registros listos para procesar: {len(datos)}")

# OPERACIONES MATEMÁTICAS ADICIONALES
# Estadísticas base
total_ventas, promedio, maximo, minimo = obtener_estadisticas(datos)

# Resta: Diferencia entre la venta más alta y la más baja
diferencia_max_min = maximo - minimo

# Multiplicación: Proyección de ventas al doble
proyeccion_doble = total_ventas * 2

# División: Calcular el aporte promedio de cada registro al total
aporte_por_registro = total_ventas / len(datos)

print("\n--- ESTADÍSTICAS ADICIONALES ---")
print(f"Suma Total de Ventas: ${total_ventas:,.2f}")          # Suma
print(f"Diferencia (Max - Min): ${diferencia_max_min:,.2f}")   # Resta
print(f"Proyección Doble: ${proyeccion_doble:,.2f}")           # Multiplicación
print(f"Aporte promedio por fila: ${aporte_por_registro:,.2f}") # División

# FILTRADO Y MUESTRA POR CATEGORÍA
# Filtra los datos para mostrar solo las ventas de una categoría específica
categoria_especifica = 'Beauty'
datos_filtrados_final = filtrar_por_categoria(datos, categoria_especifica)

if len(datos_filtrados_final) > 0:
    print(f"\n--- MOSTRANDO VENTAS PARA: {categoria_especifica} ---")
    
    # Recorrer lista para mostrar los datos en pantalla
    for venta in datos_filtrados_final:
        print(venta)
        
    print(f"\nSe encontraron {len(datos_filtrados_final)} registros de esta categoría.")
else:
    print(f"\nNo se encontraron registros para {categoria_especifica}")

# Dimensiones finales
print(f"\nDimensiones finales del arreglo (Filas, Columnas): {datos.shape}")